# 🎸 Empório AI — Customer Service Agent (Google Colab)

Artefact Brasil technical challenge — *AI Engineer, Full-Stack for Generative AI Applications* · Candidate: **Vinicius de Souza Santos**

Repository: https://github.com/ViniciusKanh/emporio-da-musica-ai-agent

This notebook clones the public repository, installs the dependencies, runs the agent on the **synthetic public
fixtures** (or on the original challenge files if you upload them), executes the automated tests and, optionally,
uses **Gemini with native function calling** and launches the **Gradio** chat.

**Run:** `Runtime → Run all`. No GPU and no API key are required (offline deterministic mode is used without a key).

In [1]:
# 1. Get the code (skipped when the notebook is already running inside the repository)
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/ViniciusKanh/emporio-da-musica-ai-agent.git"
here = Path.cwd()
candidates = [here, here.parent, Path("/content/emporio-da-musica-ai-agent")]
ROOT = next((c for c in candidates if (c / "src" / "emporio_ai" / "agent.py").exists()), None)
if ROOT is None:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/emporio-da-musica-ai-agent"], check=True)
    ROOT = Path("/content/emporio-da-musica-ai-agent")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

Project root: /home/claude/repo


In [2]:
# 2. Install dependencies (pandas, pypdf, google-genai, gradio, pytest)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-r", "requirements.txt"],
               check=True, stderr=subprocess.DEVNULL)
print("Dependencies installed")

Dependencies installed


## 3. Data source

The original challenge files are **not published** in the repository (the policy manual is marked as internal /
restricted use and `customers.csv` contains personal data, even if fictional). The repository ships **synthetic
fixtures** in `data/sample/` so everything runs publicly.

To use the original files, set `UPLOAD_ORIGINAL_FILES = True` and upload the six CSVs and `políticas_da_loja.pdf`.

In [3]:
UPLOAD_ORIGINAL_FILES = False  # set True to upload the original challenge files into data/

if UPLOAD_ORIGINAL_FILES:
    from google.colab import files  # only available on Colab
    uploaded = files.upload()
    for name, content in uploaded.items():
        (ROOT / "data" / Path(name).name).write_bytes(content)
    print("Saved:", list(uploaded))

In [4]:
# 4. Optional Gemini key: Colab -> 🔑 Secrets -> add GEMINI_API_KEY (never paste keys into the notebook)
try:
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY")
    if key:
        os.environ["GEMINI_API_KEY"] = key
except Exception:
    pass
print("Gemini key configured:", bool(os.getenv("GEMINI_API_KEY")))

Gemini key configured: False


In [5]:
# 5. Create the agent
import logging
logging.basicConfig(level=logging.WARNING)
from emporio_ai.agent import EmporioAgent
from emporio_ai.config import Settings

agent = EmporioAgent(Settings.from_env())
print("Mode:", agent.mode)
print("Data:", "synthetic sample fixtures" if agent.settings.uses_sample_data else agent.settings.data_dir)
print("Policy document:", agent.settings.policy_path.name, "| sections indexed:", len(agent.policies.chunks))
print("Rows:", agent.store.report.row_counts)
print("\nData-quality findings:")
for issue in agent.store.report.issues:
    print(" -", issue)
print("Policy conflicts:", agent.policies.store_info.conflicts)

Mode: offline


Data: synthetic sample fixtures
Policy document: policies.md | sections indexed: 21
Rows: {'products': 15, 'categories': 9, 'promotions': 6, 'orders': 5, 'order_items': 5, 'customers': 4}

Data-quality findings:
 - categories without any product in the snapshot: Instrumentos de Sopro (Madeiras), Instrumentos de Sopro (Metais), Cordas Orquestrais
 - promotions: products with more than one active promotion - only the largest discount is applied (promotions are not cumulative)
 - promotions.csv has no start/end dates: validity is taken from is_active only
 - products: 2 non-active product(s) still show stock; they are never offered as available
 - orders: recorded total differs from current list prices for order(s) [1005]; the recorded total is reported as-is
Policy conflicts: ['whatsapp: the manual lists more than one WhatsApp number: (00) 3000-0000, (00) 3000-0001']


## 6. Demonstration (each answer shows mode, tools and sources)

In [6]:
def demo(question, session="demo"):
    r = agent.ask(question, session_id=session)
    print(f"👤 {question}\n🎸 [{r.mode}] {r.text}\n   intents={r.intents}\n   tools={[c['name'] for c in r.tool_calls]}\n   sources={r.sources}\n")
    return r

if agent.settings.uses_sample_data:
    questions = ["Quais violões disponíveis até R$ 1000?", "Quanto custa o Vento VD-90E? E no PIX?",
                 "Me arrependi da minha compra, posso devolver?", "How much is the Ipê UK-30 and is it on sale?",
                 "Vocês vendem palhetas?", "Qual a previsão do tempo amanhã?"]
    order_msgs = ["Onde está meu pedido 1002?", "cliente.dois@example.com"]
else:
    questions = ["Quais opções de violões disponíveis custando até R$1000?", "Qual o endereço da loja?",
                 "Quanto custa o Takamine GD20?", "Me arrependi da minha compra, posso devolver meu pedido?",
                 "How much is the Taylor 110e and is it in stock?", "Vocês vendem palhetas?"]
    customers = agent.store.table("customers")
    order_msgs = ["Onde está meu pedido 8?", customers.loc[customers.customer_id == 2, "email"].iloc[0]]

for q in questions:
    demo(q, session=q)
# Multi-turn order verification (the e-mail is not printed)
demo(order_msgs[0], session="order")
r = agent.ask(order_msgs[1], session_id="order")
print("👤 [e-mail redacted]\n🎸", r.text)

👤 Quais violões disponíveis até R$ 1000?
🎸 [offline] Estas são as opções de **violões** disponíveis até R$ 1.000,00:
- **Aurora C10 Nylon Natural** — R$ 480,00
- **Aurora F20 Aço Natural** — R$ 690,00
Obs.: 2 item(ns) que atenderiam à busca estão sem estoque ou fora de linha, por isso não aparecem na lista.
Preços já consideram promoções vigentes. No PIX há 5% de desconto em itens fora de promoção.

Posso ajudar em mais alguma coisa?
   intents=['availability', 'product_search']
   tools=['search_products']
   sources=['products.csv', 'categories.csv', 'promotions.csv']

👤 Quanto custa o Vento VD-90E? E no PIX?
🎸 [offline] **Vento VD-90E Elétrico Aço Sunburst** (Violões)
- ✅ Disponível (3 em estoque)
- De ~~R$ 2.490,00~~ por **R$ 2.116,50** (15% off — Aniversário da Loja)
- O desconto do PIX não se aplica sobre preço promocional (promoções não são cumulativas).
- No cartão: até 12x de R$ 176,37 sem juros

Posso ajudar em mais alguma coisa?
   intents=['payment', 'price', 'product_info'

## 7. Automated tests

In [7]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"], capture_output=True, text=True)
print(result.stdout[-3000:])
assert result.returncode == 0, result.stderr[-2000:]

........................................................................ [ 72%]
...........................                                              [100%]
99 passed in 2.96s



## 8. Optional — Gemini with native function calling

Runs only when `GEMINI_API_KEY` is configured. The model chooses among read-only tools; every `R$` amount in the
final answer must match a tool result, otherwise the deterministic answer is returned (`mode = offline` + warning).

In [8]:
if agent.mode == "gemini":
    for q in ["Quanto custa o Takamine GD20?", "Quais promoções estão ativas e posso pagar com PIX com desconto?",
              "Me arrependi da compra, posso devolver?"]:
        r = agent.ask(q, session_id="gemini-" + q)
        print(f"👤 {q}\n🤖 [{r.mode}] {r.text}\n   tools={[c['name'] for c in r.tool_calls]} warnings={r.warnings}\n")
else:
    print("Skipped: no GEMINI_API_KEY configured (agent running in offline deterministic mode).")

Skipped: no GEMINI_API_KEY configured (agent running in offline deterministic mode).


## 9. Optional — Gradio chat (public temporary link on Colab)

In [9]:
LAUNCH_GRADIO = False  # set True to open the chat UI
if LAUNCH_GRADIO:
    import gradio as gr
    from app import build_ui, CSS
    build_ui(agent).launch(share=True, css=CSS, theme=gr.themes.Soft(primary_hue="pink", secondary_hue="blue"))
else:
    print("Gradio not launched (set LAUNCH_GRADIO = True).")

Gradio not launched (set LAUNCH_GRADIO = True).
